# Clipzi AI — Whisper model → Backblaze B2

Notebook ini **self-contained**: tidak perlu clone repo Clipzi yang private.

1. Isi kredensial Backblaze B2 di sel konfigurasi (atau tempel saat diminta).
2. Klik **Runtime → Run all**.
3. Setelah selesai, kembali ke admin **Settings → Models**, pilih provider
   **Backblaze B2**, lalu tekan **Scan** dan **Aktifkan** model.

Upload memakai kuota/akun Backblaze B2 milik Anda sendiri.


In [ ]:
# @title Konfigurasi model { display-mode: "form" }
MODEL_REPO = "Systran/faster-whisper-large-v3"  # @param ["Systran/faster-whisper-large-v3", "Systran/faster-whisper-large-v3-turbo", "Systran/faster-whisper-medium", "Systran/faster-whisper-small", "openai/whisper-large-v3"]
SLUG = "large-v3"  # @param {type:"string"}
QUANTIZATION = "float16"  # @param ["float16", "int8", "int8_float16"]
OVERWRITE = False  # @param {type:"boolean"}
RESUME = True  # @param {type:"boolean"}

ENDPOINT = "https://s3.us-west-004.backblazeb2.com"  # @param {type:"string"}
BUCKET = ""  # @param {type:"string"}
REGION = "us-west-004"  # @param {type:"string"}
BUCKET_ROOT = "whisper_models"  # @param {type:"string"}
PATH_STYLE = True  # @param {type:"boolean"}



In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-whisper",
        "ctranslate2",
        "transformers",
        "huggingface_hub",
        "tqdm",
        "boto3",
    ]
)


In [ ]:
from getpass import getpass

import boto3
from botocore.config import Config

ACCESS_KEY = getpass("Access key ID: ")
SECRET_KEY = getpass("Secret access key: ")

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT or None,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path" if PATH_STYLE else "auto"},
    ),
)
client.head_bucket(Bucket=BUCKET)
print("Bucket siap:", BUCKET)


In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download

WORK = Path("/content/clipzi_model")
SOURCE = WORK / "source"
CT2_DIR = WORK / "ct2"
MODEL_DIR = WORK / "upload"

if WORK.exists():
    shutil.rmtree(WORK)
SOURCE.mkdir(parents=True, exist_ok=True)

print("Mengunduh:", MODEL_REPO)
snapshot_download(repo_id=MODEL_REPO, local_dir=str(SOURCE))
print("Download selesai:", SOURCE)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REQUIRED = ("model.bin", "config.json", "tokenizer.json", "preprocessor_config.json")
VOCAB = ("vocabulary.json", "vocabulary.txt")
PYTORCH = ("model.safetensors", "pytorch_model.bin")


def names(path: Path) -> set[str]:
    return {item.name.lower() for item in path.iterdir() if item.is_file()}


found = names(SOURCE)
print("File HF:", sorted(found)[:12])

if "model.bin" in found:
    TARGET = SOURCE
else:
    if not any(marker in found for marker in PYTORCH):
        raise SystemExit(
            "Repo tidak berisi model.bin (CT2) maupun bobot PyTorch. "
            "Pakai repo Systran/faster-whisper-* atau openai/whisper-*."
        )
    print("Konversi PyTorch -> CTranslate2 (float16)…")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "ctranslate2.converters.transformers",
            "--model",
            str(SOURCE),
            "--quantization",
            QUANTIZATION,
            "--output_dir",
            str(CT2_DIR),
        ]
    )
    TARGET = CT2_DIR

present = names(TARGET)
missing = [name for name in REQUIRED if name not in present]
if missing:
    raise SystemExit(f"Model CT2 tidak lengkap, kurang: {missing}")
if not any(name in present for name in VOCAB):
    raise SystemExit("Model CT2 tidak punya vocabulary.json/vocabulary.txt.")

size = sum(item.stat().st_size for item in TARGET.iterdir() if item.is_file())
print(f"Model siap: {TARGET} ({size / (1024**3):.2f} GiB)")


In [ ]:
import boto3
from botocore.config import Config
from pathlib import Path

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT or None,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path" if PATH_STYLE else "auto"},
    ),
)

prefix = f"{BUCKET_ROOT.strip('/')}/{SLUG}" if BUCKET_ROOT.strip("/") else SLUG
for path in sorted(TARGET.iterdir()):
    if not path.is_file():
        continue
    key = f"{prefix}/{path.name}"
    print("upload", key)
    client.upload_file(
        str(path),
        BUCKET,
        key,
        ExtraArgs={"ContentType": "application/octet-stream"},
    )
print(f"Selesai: s3://{BUCKET}/{prefix}")
